# Step 1

In [16]:
MOVIES_PATH = "../data/data/movieLens_1M/movies.dat"
RATINGS_PATH = "../data/data/movieLens_1M/ratings.dat"
USER_PATH = "../data/data/movieLens_1M/users.dat"

In [6]:
import pandas as pd
import numpy as np 


In [13]:
# Load in the datasets
movies_df = pd.read_csv(
    MOVIES_PATH, 
    sep="::", 
    header=None, 
    names=["MovieID", "Title", "Genres"], 
    engine="python",
    encoding="latin-1"
)

movies_df.head()

,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [15]:
ratings_df = pd.read_csv(
    RATINGS_PATH,
    sep="::",
    header=None,
    names=["UserID", "MovieID", "Rating", "Timestamp"],
    engine="python",
)

ratings_df.head()

,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [17]:
users_df = pd.read_csv(
    USER_PATH,
    sep="::",
    header=None,
    names=["UserID", "Gender", "Age", "Occupation", "Zip-code"],
    engine="python",
)

users_df.head()

,UserID,Gender,Age,Occupation,Zip-code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [18]:
# Rating matrix
rating_matrix = ratings_df.pivot_table(
    index="UserID", 
    columns="MovieID", 
    values="Rating"
)

rating_matrix.head()

MovieID,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
UserID,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# Number of known ratings for per movie
known_movie_counts = rating_matrix.notna().sum()
print(known_movie_counts)

MovieID
1       2077
2        701
3        478
4        170
5        296
        ... 
3948     862
3949     304
3950      54
3951      40
3952     388
Length: 3706, dtype: int64


In [29]:
# Get movie with max ratings
most_rated_movie_id = known_movie_counts.idxmax()
movie_rated_movie_count = known_movie_counts.max()
most_rated_movie_title = movies_df[movies_df["MovieID"] == most_rated_movie_id]["Title"].values[0]

print(f"The most rated movie is '{most_rated_movie_title}'") 
print(f"the the most rated movie id is {most_rated_movie_id}")
print(f"The most rated movie has the rating of {movie_rated_movie_count} ratings.")

The most rated movie is 'American Beauty (1999)'
the the most rated movie id is 2858
The most rated movie has the rating of 3428 ratings.


# Step 2

In [32]:
# Y == Column of the top movie
Y = rating_matrix[most_rated_movie_id]
Y.head()

UserID
1    NaN
2    4.0
3    4.0
4    NaN
5    4.0
Name: 2858, dtype: float64

In [33]:
# X == All other movies
X = rating_matrix.drop(columns=[most_rated_movie_id])
X.head()

MovieID,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
UserID,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
# Keep only rows where Y is not NaN
non_nan_indices = Y.notna()
X = X[non_nan_indices]
Y = Y[non_nan_indices]

In [35]:
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

X shape: (3428, 3705)
Y shape: (3428,)


# Step 3

In [43]:
# Apply 3 thresholds to Y
Y = (Y > 3).astype(int)

Y.value_counts()

2858
0    3428
Name: count, dtype: int64

In [38]:
# the portion of recommended
portion_recommended = Y.mean()
print(f"Portion of users that recommended the movie: {portion_recommended}")

Portion of users that recommended the movie: 0.8322637106184364


In [39]:
# Spasity of X
num_missing = X.isna().sum().sum()
total_values_of_x = X.shape[0] * X.shape[1]
sparsity = num_missing / total_values_of_x

print(f"Sparsity of X: {sparsity}")

Sparsity of X: 0.9437629618431682
